# Cox Proportional Hazards Model on Clinical Covariates

In [ ]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict

import seaborn as sns
import matplotlib.pyplot as plt

from lifelines import CoxPHFitter, KaplanMeierFitter
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw
from sksurv.util import Surv


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def prepare_df_clinical_variables(data_df, with_grade):
    """ Setup dataset with the clinical covariates. """
    list_covariates = ['birth_days_to', 'dss_survival_days', 'dss_censorship', 'sex', 'ajcc_pathologic_tumor_stage']
    column_dict = {"birth_days_to": "age", "dss_survival_days": "survival_time", "dss_censorship": "event", "ajcc_pathologic_tumor_stage": "stage"}
    
    if with_grade:
        list_covariates.append('histological_grade')
        column_dict['histological_grade'] = 'grade'
    data_df= data_df[list_covariates]

    # Mapping dictionary
    # Other option is to do 1, 2, 3, 5, 6, 7,.. etc
    stage_mapping = {
        'Stage I': 1,
        'Stage IA': 1,
        'Stage IB': 1,
        'Stage II': 2,
        'Stage IIA': 2,
        'Stage IIB': 2,
        'Stage III': 3,
        'Stage IIIA': 3,
        'Stage IIIB': 3,
        'Stage IV': 4,
        'Stage X': np.nan,
        '[Discrepancy]': np.nan,
        '[Not Available]': np.nan,
    }

    data_df = data_df.rename(columns=column_dict)

    # Map the column
    data_df['stage'] = data_df['stage'].map(stage_mapping)
    data_df['event'] = data_df['event'].apply(lambda x: 0 if x == 1 else 1)
    data_df['age'] = data_df['age'] * -1 
    data_df['sex'] = data_df['sex'].apply(lambda x: 0 if x == 'F' else 1)

    if with_grade:
        grade_mapping = {
        'Low Grade': 1,
        'High Grade': 2,
        'G1' : 1,
        'G2' : 2,
        'G3' : 3,
        'G4' : 4,
        'GX' : np.nan, 
        '[Not Available]': np.nan, 
        '[Unknown]': np.nan}

        data_df['grade'] = data_df['grade'].map(grade_mapping)

    return data_df



In [ ]:
def coxPF_fit(data, data_test, label="survival_time", event="event"):
    """ Fit and test the cox PH model. """
    # Fit the Cox proportional hazards model with all covariates
    cph = CoxPHFitter()

    data = data.dropna()
    data_test = data_test.dropna()

    # Fit the model
    cph.fit(data, duration_col=label, event_col=event)

    # Predictions
    actual_times = data_test['survival_time']
    actual_events = data_test['event']
    predicted_scores = -cph.predict_partial_hazard(data_test)

    # c_index
    c_index = concordance_index_censored(
        actual_events.astype(bool), actual_times, predicted_scores
    )[0]

    # C-index IPCW (requires sksurv format)
    y_test_struct = Surv.from_arrays(event=actual_events.astype(bool), time=actual_times)
    y_train_struct = Surv.from_arrays(event=data[event].astype(bool), time=data[label])
    c_index_ipcw = concordance_index_ipcw(y_train_struct, y_test_struct, predicted_scores)[0]


    return {
        "c_index": c_index,
        "c_index_ipcw": c_index_ipcw,
        "actual_times": actual_times,
        "actual_events": actual_events,
        "predicted_scores": predicted_scores
    }

def CoxPH_per_fold(fold, data_source, keys):
    """ Computes test c-index of fitted cox PH models of one fold. """
    fold_folder = os.path.join(data_source, f'{fold}')

    model_types = []
    with_grade = False
    for item in keys:
        covariates = item.split('_')
        if "grade" in covariates:
            with_grade = True
        model_types.append(covariates)

    # Prepare dataset train data
    split_file_train = os.path.join(fold_folder, 'train_filtered.csv')
    data_df = pd.read_csv(split_file_train)
    clinical_df_train = prepare_df_clinical_variables(pd.DataFrame(data_df), with_grade)

    # Prepare dataset test data
    split_file_test = os.path.join(fold_folder, 'test_filtered.csv')
    data_df_test = pd.read_csv(split_file_test)
    clinical_df_test = prepare_df_clinical_variables(pd.DataFrame(data_df_test), with_grade)
    
    final_results = {}
    for item in model_types:
        covariates = item + ['survival_time', 'event']
        results_model = coxPF_fit(clinical_df_train[covariates], clinical_df_test[covariates])
        final_results['_'.join(item)] = results_model

    return final_results



def compute_clinical_results(data_source_path, with_grade=False):
    """ Computes test c-index of fitted cox PH models of all folds. """
    c_index_dict = {
        "sex_age": [],
        "sex_age_stage": []
    }
    c_index_adj_dict = {
        "sex_age": [],
        "sex_age_stage": []
    }

    if with_grade:
        c_index_dict["sex_age_grade"] = []
        c_index_adj_dict["sex_age_grade"] = []
        c_index_dict["sex_age_grade_stage"] = []
        c_index_adj_dict["sex_age_grade_stage"] = []

    pooled_km_data = {key: {"times": [], "events": [], "scores": []} for key in c_index_dict}

    for fold in range(5):

        results = CoxPH_per_fold(fold, data_source_path, list(c_index_dict.keys()))

        for key in c_index_dict:
            # Collect C-index
            c_index_dict[key].append(results[key]["c_index"])
            c_index_adj_dict[key].append(results[key]["c_index_ipcw"])
            
            # Collect pooled data for Kaplan-Meier
            pooled_km_data[key]["times"].append(results[key]["actual_times"])
            pooled_km_data[key]["events"].append(results[key]["actual_events"])
            pooled_km_data[key]["scores"].append(results[key]["predicted_scores"])



    print('Results of the clinical covariate cox PD model.')
    for key in c_index_dict:
        print("Type: ", key)
        c_index_values = np.array(c_index_dict[key])
        c_index_adj_values = np.array(c_index_adj_dict[key])
        print(f"C-index: {np.mean(c_index_values):.3f}±{np.std(c_index_values):.3f}")
        print(f"C-index IPCW: {np.mean(c_index_adj_values):.3f}±{np.std(c_index_adj_values):.3f}")

        # Plot Kaplan-Meier curves for pooled data
    return pooled_km_data


# Compute CoxPH results on all datasets

In [ ]:
all_types_km_data = {}
for with_grade, data_type in zip([False, True, False, True], ['tcga_brca', 'tcga_blca', 'tcga_luad', 'tcga_kirc']):
    print(f"\nData type: {data_type}")
    data_source = f'../data/data_files/{data_type}/splits/'
    pooled_km_data = compute_clinical_results(data_source, with_grade=with_grade)
    all_types_km_data[data_type] = pooled_km_data

# KM Curves

In [ ]:
def reverse_nested_dict(d):
    reversed_dict = defaultdict(dict)
    for outer_k, inner_d in d.items():
        for inner_k, value in inner_d.items():
            reversed_dict[inner_k][outer_k] = value
    return dict(reversed_dict)

def km_log_rank(all_data, type, save_fig):
    n_datasets = len(all_data)
    type_name = 'Cox PH ' + ' '.join(word.capitalize() for word in type.split('_'))
    fig, axes = plt.subplots(nrows=1, ncols=n_datasets, figsize=(6 * n_datasets, 5), sharex=True)

    for ax, (dataset_name, pooled_km_data) in zip(axes, all_data.items()):
        times = pd.concat(pooled_km_data["times"], ignore_index=True)
        events = pd.concat(pooled_km_data["events"], ignore_index=True)
        scores = pd.concat(pooled_km_data["scores"], ignore_index=True)   
        
        median_score = np.median(scores)
        high_risk = scores > median_score
        low_risk = scores < median_score   

        cph = CoxPHFitter()
        df = pd.DataFrame({
            "time": np.concatenate([times[high_risk], times[low_risk]]),
            "event": np.concatenate([events[high_risk], events[low_risk]]),
            "group": np.concatenate([np.ones(high_risk.sum()), np.zeros(low_risk.sum())]) 
        })
        cph.fit(df, duration_col="time", event_col="event")

        hr = cph.hazard_ratios_["group"]
        p_val = cph.summary.loc["group", "p"]

        # KM plot
        kmf = KaplanMeierFitter()
        for group, label_name, colour in zip([high_risk, low_risk], ["High Risk", "Low Risk"], ['red', 'blue']):
            kmf.fit(times[group], events[group], label=label_name)
            kmf.plot(ci_show=True, color=colour, show_censors=True, linewidth=2, ax=ax)
        ax.text(
            0.10, 0.10,
            f"HR = {hr:.3f}\np = {p_val:.2e}",
            transform=ax.transAxes,
            bbox=dict(facecolor='white', edgecolor='white'),
            fontsize=10
        )
        ax.set_ylim(0, 1.1)

        # Axis labels and title

        ax.set_ylabel("Disease-specific survival probability")
        ax.set_xlabel("Time (days)")
        ax.set_title(dataset_name.upper().replace("_", " "), fontsize=15, fontweight='bold')  # No individual subplot title
        ax.legend(loc='upper right', ncol=2, frameon=False,  fontsize=10)

    fig.text(
        -0.01, 0.5,
        type_name,
        va='center', rotation='vertical',
        fontsize=15, fontweight='bold'
    )
    plt.tight_layout()
    if save_fig:
        plt.savefig(f"../results/KM_curves/KM_{type}.pdf", format="pdf", bbox_inches="tight")
    plt.show()
    
def plot_km_curves(pooled_km_data, save_fig):
    reversed_dict = reverse_nested_dict(pooled_km_data)
    for key in reversed_dict:
        km_log_rank(reversed_dict[key], key, save_fig)


In [ ]:
save_fig = False
plot_km_curves(all_types_km_data, save_fig)